In [4]:
# SECTION 1.1 — LOAD ORIGINAL DATASET

import pandas as pd
import numpy as np

print("=" * 70)
print("LOADING ORIGINAL DATASET")
print("=" * 70)

# Update this path if your dataset has a different location
DATA_PATH = "../../dataset/mesogeos_wildfire_dataset.csv"

df = pd.read_csv(DATA_PATH)

print("\nDataset loaded successfully.")

print("\nDataset shape:")
print(f"Rows    : {df.shape[0]:,}")
print(f"Columns : {df.shape[1]:,}")

LOADING ORIGINAL DATASET

Dataset loaded successfully.

Dataset shape:
Rows    : 11,305
Columns : 32


In [5]:
# SECTION 1.2 — INSPECT DATASET COLUMNS

print("=" * 70)
print("DATASET COLUMNS")
print("=" * 70)

for i, column in enumerate(df.columns, start=1):
    print(f"{i:3d}. {column}")

print("\nTotal columns:", len(df.columns))

DATASET COLUMNS
  1. date
  2. latitude
  3. longitude
  4. temperature_c
  5. dew_point_c
  6. relative_humidity
  7. wind_speed
  8. wind_direction
  9. rainfall_mm
 10. surface_pressure
 11. solar_radiation
 12. ndvi
 13. lai
 14. soil_moisture
 15. elevation
 16. slope_degrees
 17. aspect
 18. curvature
 19. roads_distance_km
 20. population
 21. lc_agriculture
 22. lc_forest
 23. lc_grassland
 24. lc_settlement
 25. lc_shrubland
 26. lc_sparse_vegetation
 27. lc_water_bodies
 28. lc_wetland
 29. burned_area_ha
 30. year
 31. month
 32. day_of_year

Total columns: 32


In [6]:
# ============================================================
# SECTION 1.3 — DATA TYPES AND SAMPLE
# ============================================================

print("=" * 70)
print("DATASET INFORMATION")
print("=" * 70)

print("\nData types:")
display(
    df.dtypes.to_frame("Data Type")
)

print("\nFirst 5 observations:")
display(df.head())

DATASET INFORMATION

Data types:


,Data Type
date,str
latitude,float64
longitude,float64
temperature_c,float64
dew_point_c,float64
relative_humidity,float64
wind_speed,float64
wind_direction,float64
rainfall_mm,float64
surface_pressure,float64



First 5 observations:


,date,latitude,longitude,temperature_c,dew_point_c,relative_humidity,wind_speed,wind_direction,rainfall_mm,surface_pressure,...,lc_grassland,lc_settlement,lc_shrubland,lc_sparse_vegetation,lc_water_bodies,lc_wetland,burned_area_ha,year,month,day_of_year
0,2006-08-27,38.123648,-7.819233,36.419031,15.989252,22.510025,3.511700,329.388733,0.000000,99694.617188,...,0.0,0.0,0.000000,0.0,0.0,0.0,40.0,2006,8,239
1,2006-08-11,42.000202,0.447392,27.559045,7.369226,16.027991,6.567619,296.081879,0.004796,93547.523438,...,0.0,0.0,0.192824,0.0,0.0,0.0,1390.0,2006,8,223
2,2006-07-11,41.617581,-8.524061,34.471796,20.400690,35.828364,2.905688,345.955994,0.550396,99709.054688,...,0.0,0.0,0.000000,0.0,0.0,0.0,173.0,2006,7,192
3,2006-09-07,41.557167,-7.567509,30.666193,12.939386,28.009099,2.133366,313.057190,2.396404,93731.859375,...,0.0,0.0,0.000000,0.0,0.0,0.0,220.0,2006,9,250
4,2006-09-08,41.547098,-6.248474,31.690332,14.382166,22.408468,3.914824,288.407166,0.974770,93459.937500,...,0.0,0.0,0.000000,0.0,0.0,0.0,76.0,2006,9,251


In [7]:
# ============================================================
# SECTION 1.4 — SEARCH FOR DATE/TIME VARIABLES
# ============================================================

print("=" * 70)
print("DATE / TIME VARIABLE SEARCH")
print("=" * 70)

date_keywords = [
    "date",
    "time",
    "datetime",
    "year",
    "month",
    "day",
    "timestamp"
]

possible_date_columns = [
    column
    for column in df.columns
    if any(
        keyword in column.lower()
        for keyword in date_keywords
    )
]

print("\nPotential temporal columns:")

for column in possible_date_columns:
    print("✓", column)

if not possible_date_columns:
    print("No obvious date/time column found.")

DATE / TIME VARIABLE SEARCH

Potential temporal columns:
✓ date
✓ year
✓ month
✓ day_of_year


## Interpretation: Original Dataset Structure

The original Mesogeos dataset contains 32 variables, including an explicit date field, geographic coordinates, meteorological variables, vegetation characteristics, topographic variables, land-cover information, human-accessibility indicators, and the burned-area target.

The presence of an observation-level date is particularly important for dataset enhancement because it provides a temporal reference for each environmental observation. This creates the possibility of incorporating antecedent weather and environmental conditions rather than relying only on conditions measured at the observation date.

The dataset therefore provides a suitable foundation for developing temporally informed features. However, repeated observations at the exact same geographic coordinates are relatively limited, so historical features should not be created by simply assuming that nearby or unrelated observations represent the same location.

Instead, the date and geographic coordinates can be used to investigate whether external historical weather or drought information can be safely joined to each observation.

The current dataset will therefore be retained as the baseline dataset, while additional temporal and environmental information will be investigated as an enhancement.

In [9]:
# TEMPORAL COVERAGE AUDIT

print("=" * 70)
print("TEMPORAL COVERAGE AUDIT")
print("=" * 70)

# Convert date to datetime
df["date"] = pd.to_datetime(
    df["date"],
    errors="coerce"
)

print("\nDate conversion completed.")

print("\nMissing dates:")
print(df["date"].isna().sum())

print("\nEarliest observation:")
print(df["date"].min())

print("\nLatest observation:")
print(df["date"].max())

print("\nTotal days represented:")
print(
    (df["date"].max() - df["date"].min()).days
)

print("\nUnique dates:")
print(df["date"].nunique())

print("\nObservations per date:")
display(
    df.groupby("date")
      .size()
      .describe()
      .round(2)
      .to_frame("Observations")
)

TEMPORAL COVERAGE AUDIT

Date conversion completed.

Missing dates:
0

Earliest observation:
2006-04-09 00:00:00

Latest observation:
2022-09-28 00:00:00

Total days represented:
6016

Unique dates:
2819

Observations per date:


,Observations
count,2819.00
mean,4.01
std,4.12
min,1.00
25%,1.00
50%,2.00
75%,5.00
max,30.00


In [12]:
# ============================================================
# SECTION 2.2 — YEAR AND MONTH COVERAGE
# ============================================================

print("=" * 70)
print("YEAR AND MONTH COVERAGE")
print("=" * 70)

print("\nObservations by year:")

year_counts = (
    df.groupby("year")
      .size()
      .reset_index(name="Observations")
)

display(year_counts)

print("\nObservations by month:")

month_counts = (
    df.groupby("month")
      .size()
      .reset_index(name="Observations")
)

display(month_counts)

YEAR AND MONTH COVERAGE

Observations by year:


,year,Observations
0,2006,214
1,2007,603
2,2008,296
3,2009,456
4,2010,420
5,2011,781
6,2012,1046
7,2013,424
8,2014,320
9,2015,460



Observations by month:


,month,Observations
0,1,157
1,2,375
2,3,1025
3,4,411
4,5,198
5,6,707
6,7,2761
7,8,3587
8,9,1313
9,10,545


In [13]:
# ============================================================
# SECTION 2.3 — LOCATION-TIME REPETITION
# ============================================================

print("=" * 70)
print("LOCATION-TIME REPETITION")
print("=" * 70)

location_date_counts = (
    df.groupby(
        ["latitude", "longitude"]
    )
    .agg(
        observations=("date", "size"),
        unique_dates=("date", "nunique"),
        first_date=("date", "min"),
        last_date=("date", "max")
    )
    .reset_index()
)

repeated = location_date_counts[
    location_date_counts["observations"] > 1
].copy()

print("\nRepeated locations:")
print(len(repeated))

print("\nRepeated-location observation counts:")
display(
    repeated["observations"]
    .value_counts()
    .sort_index()
    .to_frame("Number of Locations")
)

print("\nExample repeated locations:")

display(
    repeated
    .sort_values(
        "observations",
        ascending=False
    )
    .head(10)
)

LOCATION-TIME REPETITION

Repeated locations:
313

Repeated-location observation counts:


,Number of Locations
observations,
2,301
3,12



Example repeated locations:


,latitude,longitude,observations,unique_dates,first_date,last_date
2059,37.167095,14.110987,3,3,2007-09-08,2017-09-06
2144,37.227509,13.778711,3,3,2012-06-23,2018-07-19
2393,37.449027,13.476642,3,3,2010-07-23,2022-07-24
2226,37.287923,14.010297,3,3,2010-06-19,2014-07-07
2974,37.982682,15.691816,3,3,2007-07-03,2017-06-19
2452,37.499372,13.396090,3,3,2007-08-28,2018-08-06
5585,41.003374,-7.960199,3,3,2011-10-01,2016-10-05
5996,41.295374,-7.929992,3,3,2006-07-14,2020-03-10
6111,41.375926,-7.758820,3,3,2014-03-16,2019-03-09
7671,42.312340,-6.731785,3,3,2011-03-20,2022-02-06


## Interpretation: Temporal and Location-Time Structure

The temporal audit shows that the dataset covers a substantial period from April 2006 to September 2022, representing approximately 6,016 days and 2,819 unique observation dates. There are no missing dates among the recorded observations.

The number of observations per date varies considerably. The median is only 2 observations per date, while the maximum is 30 observations. This indicates that the dataset is spatially sparse on most individual dates rather than representing a continuous monitoring network.

The yearly distribution also demonstrates that observations are available across multiple years from 2006 to 2022, although the number of observations varies substantially between years. The dataset is particularly concentrated in several years, while other years contain considerably fewer observations.

The monthly distribution shows a strong seasonal concentration, with July and August containing the largest numbers of observations. This is consistent with the seasonal nature of forest-fire activity but also indicates that the dataset is temporally imbalanced.

The location-time analysis identified 313 locations with repeated observations. Of these, 301 locations occur twice and only 12 locations occur three times. Therefore, repeated observations at exactly the same geographic coordinates are relatively rare.

Importantly, repeated locations are separated by substantial periods of time. For example, some locations have observations separated by several years. Consequently, the existing dataset does not provide sufficient continuous observations at individual locations to reliably calculate local rolling weather variables such as 7-day, 14-day, or 30-day rainfall using only the existing rows.

However, the presence of an exact observation date together with latitude and longitude provides an important opportunity for dataset enhancement. Historical meteorological and drought-related information can potentially be obtained for each observation location and date from an external historical weather or reanalysis source.

Therefore, the dataset will be retained as the baseline dataset, while temporal environmental information will be added through a carefully controlled date-and-location-based data enrichment process.

## Historical Fire-Weather Feature Design

The existing dataset primarily represents environmental conditions associated with individual observations. To improve burned-area prediction, additional antecedent environmental features will be considered.

The proposed features are designed to capture the cumulative drying and fire-weather conditions preceding each observation.

### Proposed precipitation features
- 7-day accumulated rainfall
- 14-day accumulated rainfall
- 30-day accumulated rainfall

### Proposed dryness features
- Number of consecutive dry days
- Number of dry days during the previous 14 days
- Number of dry days during the previous 30 days
- Cumulative rainfall deficit

### Proposed temperature features
- 7-day mean temperature
- 14-day mean temperature
- 30-day mean temperature
- Recent maximum temperature

### Proposed humidity features
- 7-day minimum relative humidity
- 14-day minimum relative humidity
- 30-day minimum relative humidity

### Proposed wind features
- 7-day maximum wind speed
- 14-day maximum wind speed
- 30-day maximum wind speed

### Proposed moisture and drought indicators
- Recent soil-moisture condition
- Soil-moisture change
- Vapor Pressure Deficit (VPD)
- Drought-related indicators where available

All historical features must be calculated using information available before the observation date in order to prevent data leakage.

## Original Mesogeos Data Sources

The Mesogeos dataset integrates multiple environmental and wildfire-related data sources within a common 1 km × 1 km daily spatiotemporal grid covering the Mediterranean region from 2006 to 2022.

The principal data sources include:

- **ERA5-Land** for meteorological variables such as temperature, dew point, relative humidity, wind speed, surface pressure, precipitation, and solar radiation.
- **MODIS** for vegetation indicators including NDVI and LAI.
- **JRC European Drought Observatory** for soil moisture information.
- **WorldPop** for population data and road accessibility information.
- **Copernicus Climate Change Service** for land-cover information.
- **Copernicus EU-DEM** for elevation and terrain characteristics.
- **EFFIS** for wildfire ignition and burned-area information.

The official Mesogeos documentation describes the underlying dataset as a daily 1 km × 1 km datacube covering the period from 2006 to 2022.

This source structure is advantageous for the proposed dataset enhancement because historical weather and environmental conditions can potentially be extracted from the same underlying data source rather than combining unrelated external datasets.

The enhancement process will therefore attempt to use the original Mesogeos datacube to derive antecedent environmental features while ensuring that only information available before the fire observation date is used.

In [2]:
import xarray as xr

path = r"C:\Projects\ML Based Forest-Fire Prediction Project\dataset\mesogeos_dataset\2006\sample_0.nc"

ds = xr.open_dataset(path)

print(ds)

<xarray.Dataset> Size: 476kB
Dimensions:               (y: 64, x: 64, time: 1)
Coordinates:
  * y                     (y) float64 512B 38.45 38.44 38.43 ... 37.82 37.81
  * x                     (x) float64 512B -8.141 -8.131 ... -7.517 -7.507
  * time                  (time) datetime64[ns] 8B 2006-08-27
Data variables: (12/30)
    aspect                (y, x) float32 16kB ...
    burned_areas          (time, y, x) float32 16kB ...
    curvature             (y, x) float32 16kB ...
    d2m                   (time, y, x) float32 16kB ...
    dem                   (y, x) float32 16kB ...
    ignition_points       (time, y, x) float32 16kB ...
    ...                    ...
    lc_settlement         (y, x) float32 16kB ...
    lc_shrubland          (y, x) float32 16kB ...
    lc_sparse_vegetation  (y, x) float32 16kB ...
    lc_water_bodies       (y, x) float32 16kB ...
    lc_wetland            (y, x) float32 16kB ...
    population            (y, x) float32 16kB ...


In [3]:
print("Variables:")
print(list(ds.data_vars))

print("\nDimensions:")
print(ds.dims)

print("\nCoordinates:")
print(list(ds.coords))

Variables:
['aspect', 'burned_areas', 'curvature', 'd2m', 'dem', 'ignition_points', 'lai', 'lst_day', 'lst_night', 'ndvi', 'rh', 'roads_distance', 'slope', 'smi', 'sp', 'spatial_ref', 'ssrd', 't2m', 'tp', 'wind_direction', 'wind_speed', 'lc_agriculture', 'lc_forest', 'lc_grassland', 'lc_settlement', 'lc_shrubland', 'lc_sparse_vegetation', 'lc_water_bodies', 'lc_wetland', 'population']

Dimensions:
FrozenMappingWarningOnValuesAccess({'y': 64, 'x': 64, 'time': 1})

Coordinates:
['time', 'x', 'y']


In [4]:
ds.info()

xarray.Dataset {
dimensions:
	y = 64 ;
	x = 64 ;
	time = 1 ;

variables:
	float32 aspect(y, x) ;
		aspect:grid_mapping = spatial_ref ;
	float32 burned_areas(time, y, x) ;
		burned_areas:grid_mapping = spatial_ref ;
	float32 curvature(y, x) ;
		curvature:grid_mapping = spatial_ref ;
	float32 d2m(time, y, x) ;
		d2m:grid_mapping = spatial_ref ;
	float32 dem(y, x) ;
		dem:grid_mapping = spatial_ref ;
	float32 ignition_points(time, y, x) ;
		ignition_points:grid_mapping = spatial_ref ;
	float32 lai(time, y, x) ;
		lai:grid_mapping = spatial_ref ;
	float32 lst_day(time, y, x) ;
		lst_day:grid_mapping = spatial_ref ;
	float32 lst_night(time, y, x) ;
		lst_night:grid_mapping = spatial_ref ;
	float32 ndvi(time, y, x) ;
		ndvi:grid_mapping = spatial_ref ;
	float32 rh(time, y, x) ;
		rh:grid_mapping = spatial_ref ;
	float32 roads_distance(y, x) ;
		roads_distance:grid_mapping = spatial_ref ;
	float32 slope(y, x) ;
		slope:grid_mapping = spatial_ref ;
	float32 smi(time, y, x) ;
		smi:grid_mapping